In [ ]:
import sys
import os
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType

# Đảm bảo Python nhận diện thư mục hiện tại (src) để import spark_session
sys.path.append(os.path.abspath("."))


In [ ]:
import spark_session

# Tự động lấy đối tượng spark hoặc hàm khởi tạo bên trong spark_session.py
if hasattr(spark_session, "get_spark"):
    spark = spark_session.get_spark("HM_Clean_Huy")
elif hasattr(spark_session, "create_spark_session"):
    spark = spark_session.create_spark_session("HM_Clean_Huy")
elif hasattr(spark_session, "spark"):
    spark = spark_session.spark
else:
    raise AttributeError("Kiểm tra tên hàm tạo Spark bên trong file src/spark_session.py!")

print("Đã khởi tạo Spark version:", spark.version)

In [ ]:
# đọc dữ liệu từ file CSV
df_art = spark.read.parquet("../data/raw/article.parquet")
df_cust = spark.read.parquet("../data/raw/customer.parquet")
df_trans = spark.read.parquet("../data/processed/transactions.parquet")
# đếm số dòng giao dich và số khách hàng duy nhất trong DataFrame
stats = []
def log_step(step_name, df):
    row_cnt = df.count()
    cust_cnt = df.select("customer_id").distinct().count()
    stats.append({
        "Buoc": step_name,
        "So_dong": row_cnt,
        "So_khach_hang": cust_cnt
    })
    print(f"[{step_name}] Số dòng: {row_cnt:,} | Số khách hàng: {cust_cnt:,}")

log_step("0. Dữ liệu gốc ban đầu", df_trans)

In [ ]:
# chuẩn hóa dữ liệu
df_trans = (
    df_trans
    .withColumn("t_dat", F.to_date(F.col("t_dat")))
    .withColumn("price", F.col("price").cast(DoubleType()))
    .withColumn("sales_channel_id", F.col("sales_channel_id").cast(LongType()))
)

In [ ]:
# Loại dòng thiếu khóa chính và ngày giao dịch
df_trans = df_trans.dropna(subset=["customer_id", "article_id", "t_dat"])
log_step("1. Sau khi loại bỏ null khóa chính & ngày", df_trans)

# dùng left_anti join để loại bỏ các dòng giao dịch có khóa mồ côi
orphan_cust = df_trans.join(df_cust, on="customer_id", how="left_anti").count()
orphan_art = df_trans.join(df_art, on="article_id", how="left_anti").count()
print(f"-> Số dòng khóa mồ côi: customer_id = {orphan_cust:,}, article_id = {orphan_art:,}")
# dùng join để lọc các dòng giao dịch có khóa tồn tại trong cả hai bảng
df_trans = (
    df_trans
    .join(df_cust.select("customer_id"), on="customer_id", how="inner")
    .join(df_art.select("article_id"), on="article_id", how="inner")
)
log_step("2. Sau khi loại bỏ khóa mồ côi", df_trans)

In [ ]:
# Loại bỏ giao dịch có price <= 0 hoặc NaN/Null
df_trans = df_trans.filter((F.col("price").isNotNull()) & (~F.isnan("price")) & (F.col("price") > 0))
log_step("3. Sau khi lọc price > 0", df_trans)

In [ ]:
# Thống kê dòng trùng lặp hoàn toàn 
total_rows = df_trans.count()
distinct_rows = df_trans.dropDuplicates().count()
dup_rows = total_rows - distinct_rows
print(f"-> Số dòng trùng lặp hoàn toàn (được giữ lại): {dup_rows:,} ({dup_rows/total_rows*100:.2f}%)")

In [ ]:
# Làm sạch bảng customer 
median_age = df_cust.filter((F.col("age") >= 12) & (F.col("age") <= 100)).approxQuantile("age", [0.5], 0.01)[0]
print(f"-> Tuổi trung vị hợp lệ: {median_age}")

df_cust_clean = (
    df_cust
    .withColumn("is_age_missing", F.when((F.col("age").isNull()) | (F.col("age") < 12) | (F.col("age") > 100), 1).otherwise(0))
    .withColumn("age_clean", F.when(F.col("is_age_missing") == 1, median_age).otherwise(F.col("age")))
    .withColumn("club_member_status", F.coalesce(F.col("club_member_status"), F.lit("UNKNOWN")))
    .withColumn("fashion_news_frequency", F.coalesce(F.col("fashion_news_frequency"), F.lit("NONE")))
)

In [ ]:
# Lưu dữ liệu sạch như mục data/clean
os.makedirs("../data/clean", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

df_trans.write.mode("overwrite").parquet("../data/clean/transactions_clean.parquet")
df_cust_clean.write.mode("overwrite").parquet("../data/clean/customer_clean.parquet")
df_art.write.mode("overwrite").parquet("../data/clean/article_clean.parquet")

In [ ]:
# xuất bảng thống kê làm sạch 
df_stats = pd.DataFrame(stats)
df_stats.to_csv("../reports/thong_ke_lam_sach.csv", index=False, encoding="utf-8-sig")
df_stats

In [ ]:
# Đọc lại dữ liệu từ thư mục data/clean
trans_verify = spark.read.parquet("../data/clean/transactions_clean.parquet")
cust_verify = spark.read.parquet("../data/clean/customer_clean.parquet")
art_verify = spark.read.parquet("../data/clean/article_clean.parquet")

In [ ]:
# Kiểm tra kiểu dữ liệu đã chuẩn hóa chưa
print("--- 1. KIỂM TRA SCHEMA BẢNG TRANSACTIONS ---")
trans_verify.printSchema()

In [ ]:
# Kiểm tra Null / NaN và miền giá trị hợp lệ
err_trans_null = trans_verify.filter(
    F.col("customer_id").isNull() | 
    F.col("article_id").isNull() | 
    F.col("t_dat").isNull() | 
    F.col("price").isNull() | 
    F.isnan("price")
).count()

err_price = trans_verify.filter(F.col("price") <= 0).count()

err_cust_age = cust_verify.filter(
    F.col("age_clean").isNull() | 
    (F.col("age_clean") < 12) | 
    (F.col("age_clean") > 100)
).count()

err_cust_cat = cust_verify.filter(
    F.col("club_member_status").isNull() | 
    F.col("fashion_news_frequency").isNull()
).count()

In [ ]:
# Kiểm tra lại khóa mồ côi 
err_orphan_cust = trans_verify.join(cust_verify, on="customer_id", how="left_anti").count()
err_orphan_art = trans_verify.join(art_verify, on="article_id", how="left_anti").count()

print(f"- Dòng thiếu khóa chính / ngày / giá : {err_trans_null}")
print(f"- Giao dịch có giá <= 0            : {err_price}")
print(f"- Khách hàng sai độ tuổi [12-100]  : {err_cust_age}")
print(f"- Khách hàng còn Null phân loại    : {err_cust_cat}")
print(f"- Khóa mồ côi (Customer / Article) : {err_orphan_cust} / {err_orphan_art}")

In [ ]:
# Xem trước 5 dòng dữ liệu sạch của bảng Customer
cust_verify.select("customer_id", "age", "is_age_missing", "age_clean", "club_member_status", "fashion_news_frequency").show(5, truncate=False)